In [1]:
import requests
import json
import re
import pandas as pd
from difflib import SequenceMatcher

In [ ]:
API_KEY = ""

SCOPUS_SEARCH_URL = "https://api.elsevier.com/content/search/scopus"

headers = {
    "X-ELS-APIKey": API_KEY,
    "Accept": "application/json"
}

In [3]:
def normalize_title(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = text.replace("&amp;", " and ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def title_similarity(a, b):
    a_norm = normalize_title(a)
    b_norm = normalize_title(b)

    if not a_norm or not b_norm:
        return 0

    return SequenceMatcher(None, a_norm, b_norm).ratio()


def clean_title_for_query(title, max_len=280):
    title = str(title)
    title = title.replace('"', "")
    title = title.replace("“", "")
    title = title.replace("”", "")
    title = re.sub(r"\s+", " ", title).strip()
    return title[:max_len]


def extract_scopus_link(links):
    if not links:
        return None

    for link in links:
        if link.get("@ref") == "scopus":
            return link.get("@href")

    return None


def format_scopus_item(item, input_title=None):
    links = item.get("link", [])

    output = {
        "input_title": input_title,
        "matched_title": item.get("dc:title"),
        "title_similarity": title_similarity(input_title, item.get("dc:title")) if input_title else None,

        "doi": item.get("prism:doi"),
        "doi_link": f"https://doi.org/{item.get('prism:doi')}" if item.get("prism:doi") else None,

        "eid": item.get("eid"),
        "scopus_id": item.get("dc:identifier"),
        "scopus_link": extract_scopus_link(links),

        "journal_or_source": item.get("prism:publicationName"),
        "issn": item.get("prism:issn"),
        "eissn": item.get("prism:eIssn"),
        "isbn": item.get("prism:isbn"),

        "volume": item.get("prism:volume"),
        "issue": item.get("prism:issueIdentifier"),
        "page_range": item.get("prism:pageRange"),

        "cover_date": item.get("prism:coverDate"),
        "cover_display_date": item.get("prism:coverDisplayDate"),

        "citedby_count": item.get("citedby-count"),

        "aggregation_type": item.get("prism:aggregationType"),
        "subtype": item.get("subtype"),
        "subtype_description": item.get("subtypeDescription"),

        "creator": item.get("dc:creator"),
        "authors": item.get("authors"),
        "affiliation": item.get("affiliation"),

        "openaccess": item.get("openaccess"),
        "openaccess_flag": item.get("openaccessFlag"),

        "source_id": item.get("source-id"),
        "article_number": item.get("article-number"),

        "raw_item": item
    }

    return output

In [4]:
def cek_scopus_judul_lengkap(judul, count=10, timeout=30, tampilkan_raw=False):
    title_clean = clean_title_for_query(judul)

    if not title_clean:
        print("Judul kosong.")
        return None

    query = f'TITLE("{title_clean}")'

    params = {
        "query": query,
        "count": count,
        "start": 0
    }

    response = requests.get(
        SCOPUS_SEARCH_URL,
        headers=headers,
        params=params,
        timeout=timeout
    )

    print("HTTP Status:", response.status_code)
    print("Query:", query)

    if response.status_code != 200:
        print(response.text[:1000])
        return None

    data = response.json()

    if tampilkan_raw:
        print(json.dumps(data, indent=2, ensure_ascii=False))

    entries = data.get("search-results", {}).get("entry", [])

    if not entries:
        print("Tidak ada hasil dari Scopus.")
        return {
            "query": query,
            "total_results": 0,
            "results": [],
            "raw_json": data
        }

    results = [
        format_scopus_item(item, input_title=judul)
        for item in entries
    ]

    results = sorted(
        results,
        key=lambda x: x["title_similarity"] if x["title_similarity"] is not None else 0,
        reverse=True
    )

    output = {
        "query": query,
        "total_results": data.get("search-results", {}).get("opensearch:totalResults"),
        "returned_results": len(results),
        "results": results,
        "raw_json": data
    }

    return output

In [5]:
judul = "Sustainable human resource management: Strategy, organizational innovation and leadership in industry 5.0"

hasil = cek_scopus_judul_lengkap(
    judul,
    count=10,
    tampilkan_raw=False
)

hasil

HTTP Status: 200
Query: TITLE("Sustainable human resource management: Strategy, organizational innovation and leadership in industry 5.0")


{'query': 'TITLE("Sustainable human resource management: Strategy, organizational innovation and leadership in industry 5.0")',
 'total_results': '1',
 'returned_results': 1,
 'results': [{'input_title': 'Sustainable human resource management: Strategy, organizational innovation and leadership in industry 5.0',
   'matched_title': 'Sustainable human resource management: Strategy, organizational innovation and leadership in industry 5.0',
   'title_similarity': 1.0,
   'doi': '10.4324/9781003458432',
   'doi_link': 'https://doi.org/10.4324/9781003458432',
   'eid': '2-s2.0-85200960273',
   'scopus_id': 'SCOPUS_ID:85200960273',
   'scopus_link': 'https://www.scopus.com/inward/record.uri?partnerID=HzOxMe3b&scp=85200960273&origin=inward',
   'journal_or_source': 'Sustainable Human Resource Management Strategy Organizational Innovation and Leadership in Industry 5 0',
   'issn': None,
   'eissn': None,
   'isbn': [{'@_fa': 'true', '$': '[9781032598949, 9781040096246]'}],
   'volume': None,
